In [1]:
# ==========================================
# 1. IMPORTS & SETUP
# ==========================================
import pandas as pd
import json
import glob
import os
import re
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from IPython.display import display, Markdown

# Set pandas to show all columns for better inspection
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries loaded.")


Libraries loaded.


In [2]:

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def extract_eids_from_gpt_output(gpt_output):
    """Extract unique E-IDs from LLM output."""
    text_blocks = []
    if isinstance(gpt_output, dict):
        if 'why_correct' in gpt_output:
            text_blocks.append(str(gpt_output['why_correct']))
        if 'why_others_incorrect' in gpt_output:
            woi = gpt_output['why_others_incorrect']
            if isinstance(woi, dict):
                text_blocks.extend([str(v) for v in woi.values()])
    
    combined_text = " ".join(text_blocks)
    matches = re.findall(r"E\d+", combined_text)
    return list(set(matches))

def get_dominance_category(routes, ranks):
    """Classify evidence quality."""
    if not routes: return "No Evidence"
    evidence_items = list(zip(routes, ranks))
    
    kg_nodes = [item for item in evidence_items if item[0] == 'kg_sui_concept_def']
    
    if kg_nodes:
        best_kg_rank = min(r for _, r in kg_nodes)
        if best_kg_rank <= 5:
            return "KG Traversal (Top 5 - Critical)"
        else:
            return "KG Traversal (Rank >5 - Supporting)"
    
    if any(r == 'kg_semantic_name_only' for r in routes):
        return "KG Entity Match (Name)"
        
    return "Dense Retrieval (Vector)"

def normalize_evidence_source(evidence_data):
    """Normalize list or dict evidence to dict format."""
    ev_map = {}
    if isinstance(evidence_data, list):
        for item in evidence_data:
            ev_map[item['id']] = item.get('all_evidence', [])
    elif isinstance(evidence_data, dict):
        ev_map = evidence_data
    return ev_map

def determine_complexity(predicted, used_ranks):
    if predicted == "-1":
        return "Abstentions"
    elif len(used_ranks) == 1 and used_ranks[0] == 1:
        return "Simple Answers"
    else:
        return "Complex Answers"

# ==========================================
# 3. DATA LOADING & PROCESSING
# ==========================================

def load_data():
    print("Loading data from 'outputs/' folder...")
    files = glob.glob("outputs/*_model_*.json")
    all_records = []
    
    # Pre-load caches
    cache_files = glob.glob("outputs/*_retrieval_cache.json")
    caches = {}
    for c_file in cache_files:
        try:
            t_name = os.path.basename(c_file).replace("_retrieval_cache.json", "")
            with open(c_file, 'r') as f:
                caches[t_name] = json.load(f)
            print(f"✅ Loaded Cache: {t_name}")
        except: pass

    for model_file in files:
        filename = os.path.basename(model_file)
        match = re.match(r"(.+)_model_(.+)\.json", filename)
        if not match: continue
        
        task_name = match.group(1)
        model_name = match.group(2)
        
        # Evidence Logic
        raw_evidence_data = None
        
        # 1. Try Direct File
        ev_file = model_file.replace("_model_", "_evidence_")
        if os.path.exists(ev_file):
            try:
                with open(ev_file, 'r') as f: raw_evidence_data = json.load(f)
            except: pass
        
        # 2. Try Cache
        if not raw_evidence_data:
            if task_name in caches:
                raw_evidence_data = caches[task_name]
            else:
                for c_name, c_data in caches.items():
                    if c_name in task_name:
                        raw_evidence_data = c_data
                        break
        
        if not raw_evidence_data: 
            print(f"⚠️ No evidence found for {filename}. Skipping deep analysis columns.")
            continue

        ev_map = normalize_evidence_source(raw_evidence_data)

        try:
            with open(model_file, 'r') as f: m_data = json.load(f)
        except: continue

        for m_item in m_data:
            q_id = m_item['id']
            if q_id not in ev_map: continue
            
            all_nodes = ev_map[q_id]
            gpt_out = m_item.get('gpt_output', {})
            
            predicted = str(gpt_out.get('cop_index', '-1')).strip()
            correct = str(m_item.get('testbed_data', {}).get('correct_index', '')).strip()
            
            is_abstain = (predicted == "-1")
            
            if "fake" in task_name:
                is_correct = is_abstain
            else:
                is_correct = (predicted == correct)

            used_eids = extract_eids_from_gpt_output(gpt_out)
            
            all_nodes_sorted = sorted(all_nodes, key=lambda x: x['score'], reverse=True)
            node_lookup = {n['eid']: {'rank': i+1, 'route': n['route'], 'score': n['score']} 
                           for i, n in enumerate(all_nodes_sorted)}

            used_routes = []
            used_ranks = []
            used_scores = []
            
            for eid in used_eids:
                if eid in node_lookup:
                    used_routes.append(node_lookup[eid]['route'])
                    used_ranks.append(node_lookup[eid]['rank'])
                    used_scores.append(node_lookup[eid]['score'])
            
            has_kg = any('kg_sui' in r for r in used_routes)
            avg_score = np.mean(used_scores) if used_scores else 0.0
            max_score = max(used_scores) if used_scores else 0.0
            
            cat = get_dominance_category(used_routes, used_ranks)
            complexity = determine_complexity(predicted, used_ranks)
            
            # Score Profile
            score_profile = [n['score'] for n in all_nodes_sorted[:32]]
            while len(score_profile) < 32: score_profile.append(0.0)
            
            all_records.append({
                "Task": task_name,
                "Model": model_name,
                "ID": q_id,
                "Is Correct": is_correct,
                "Is Abstain": is_abstain,
                "Has KG Trace": has_kg,
                "Avg Evidence Score": avg_score,
                "Max Used Score": max_score,
                "Score Profile": score_profile,
                "Evidence Category": cat,
                "Complexity": complexity,
                "Used Ranks": used_ranks,
                "Used Routes": used_routes,
                "Used Scores": used_scores
            })
            
    return pd.DataFrame(all_records)


In [ ]:

# Load the data
df = load_data()

print(f"\n✅ Data Loaded. Total Rows: {len(df)}")
display(df.head())

# ==========================================
# 4. FILTERING (Interactive Selection)
# ==========================================

# 1. SELECT TASK
unique_tasks = sorted(df['Task'].unique())
print(f"\nAvailable Tasks: {unique_tasks}")




Loading data from 'outputs/' folder...
✅ Loaded Cache: reasoning_nota
✅ Loaded Cache: reasoning_fake_100
✅ Loaded Cache: reasoning_fct_100
✅ Loaded Cache: reasoning_fct_200
✅ Loaded Cache: reasoning_fct
✅ Loaded Cache: reasoning_fct_1
✅ Loaded Cache: reasoning_fake

✅ Data Loaded. Total Rows: 50425


,Task,Model,ID,Is Correct,Is Abstain,Has KG Trace,Avg Evidence Score,Max Used Score,Score Profile,Evidence Category,Complexity,Used Ranks,Used Routes,Used Scores
0,reasoning_nota,phi4-14b,07d9bcf9-feb6-4d4a-858e-5570c5d9ef8b,True,False,False,0.523443,0.671036,"[0.671036422252655, 0.6652645468711853, 0.4812...",Dense Retrieval (Vector),Complex Answers,"[1, 32]","[dense_def_faiss, dense_def_faiss]","[0.671036422252655, 0.3758489787578583]"
1,reasoning_nota,phi4-14b,d8c969f3-fa33-421c-afb2-62f6fc51fa9b,False,False,False,0.564825,0.676108,"[0.6942787170410156, 0.6831874251365662, 0.676...",KG Entity Match (Name),Complex Answers,"[3, 30]","[kg_semantic_name_only, dense_def_faiss]","[0.67610764503479, 0.4535427689552307]"
2,reasoning_nota,phi4-14b,8fe4e14c-4e17-4af0-94c2-0d8a59f342b8,False,False,False,0.469932,0.538255,"[0.6733906269073486, 0.6303748488426208, 0.610...",KG Entity Match (Name),Complex Answers,"[12, 32, 14, 17, 18]","[kg_semantic_name_only, dense_def_faiss, dense...","[0.538254976272583, 0.36240679025650024, 0.504..."
3,reasoning_nota,phi4-14b,6bfcd1fd-e172-4490-bf96-5da1b8da7ed4,True,False,False,0.605250,0.653766,"[0.8003321886062622, 0.7044739127159119, 0.653...",Dense Retrieval (Vector),Complex Answers,"[4, 3, 20, 7]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.6410924196243286, 0.6537659168243408, 0.512..."
4,reasoning_nota,phi4-14b,dae6278f-b7f8-42ea-b5d6-3d76082df175,True,False,False,0.615408,0.672951,"[0.8036119937896729, 0.7734134197235107, 0.764...",Dense Retrieval (Vector),Complex Answers,"[5, 8, 20]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.6729507446289062, 0.6224954724311829, 0.550..."



Available Tasks: ['Old_reasoning_fct', 'reasoning_fake', 'reasoning_fake_100', 'reasoning_fct', 'reasoning_fct_1', 'reasoning_fct_100', 'reasoning_fct_200', 'reasoning_nota']
👉 Selected Task: Old_reasoning_fct


In [4]:
# CHANGE THIS VARIABLE TO FILTER TASK
SELECTED_TASK = unique_tasks[3]  # e.g., 'reasoning_fct' or 'reasoning_fake'
print(f"👉 Selected Task: {SELECTED_TASK}")

👉 Selected Task: reasoning_fct


In [5]:

task_df = df[df['Task'] == SELECTED_TASK]
display(task_df.head())

,Task,Model,ID,Is Correct,Is Abstain,Has KG Trace,Avg Evidence Score,Max Used Score,Score Profile,Evidence Category,Complexity,Used Ranks,Used Routes,Used Scores
4458,reasoning_fct,llama3.3-70b,bc8659f4-3062-4f57-9e24-e32ad92a8d4e,False,False,False,0.777177,0.777177,"[0.7771770358085632, 0.7742263078689575, 0.761...",Dense Retrieval (Vector),Simple Answers,[1],[dense_def_faiss],[0.7771770358085632]
4459,reasoning_fct,llama3.3-70b,9fcf1ab0-387c-447a-ab6c-78367b5e5282,True,False,False,0.449315,0.599707,"[0.7846348881721497, 0.7393569946289062, 0.599...",Dense Retrieval (Vector),Complex Answers,"[6, 14, 3, 22, 21]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.5750370025634766, 0.40172672271728516, 0.59..."
4460,reasoning_fct,llama3.3-70b,0ac6c5c7-9826-441a-81d5-68478e6299bb,True,False,False,0.629167,0.777335,"[0.7773351669311523, 0.6264742612838745, 0.577...",KG Entity Match (Name),Complex Answers,"[1, 11]","[kg_semantic_name_only, kg_semantic_name_only]","[0.7773351669311523, 0.48099976778030396]"
4461,reasoning_fct,llama3.3-70b,a6758ac9-7608-4866-bb00-c5e7b19917d5,False,True,False,0.454734,0.502303,"[0.7463321089744568, 0.6520719528198242, 0.641...",Dense Retrieval (Vector),Abstentions,"[25, 24, 10]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.43044325709342957, 0.4314565062522888, 0.50..."
4462,reasoning_fct,llama3.3-70b,839de867-3100-4283-a219-ec349eee415f,True,False,True,0.477241,0.561376,"[0.8007383346557617, 0.7566207051277161, 0.732...",KG Traversal (Rank >5 - Supporting),Complex Answers,"[13, 16, 23, 6, 10, 14, 22, 17]","[dense_def_faiss, kg_sui_concept_def, dense_de...","[0.4830428957939148, 0.4739253520965576, 0.397..."


In [7]:

# 2. SELECT MODELS
unique_models = sorted(task_df['Model'].unique())
print(f"Available Models: {unique_models}")

# CHANGE THIS VARIABLE TO FILTER MODELS
SELECTED_MODELS = unique_models # Select all by default
# SELECTED_MODELS = ['phi4:14b', 'gemma2:27b'] # Uncomment to pick specific ones

filtered_df = task_df[task_df['Model'].isin(SELECTED_MODELS)]
print(f"👉 Analyzing {len(filtered_df)} rows for models: {SELECTED_MODELS}")

# ==========================================
# 5. ANALYSIS: ACCURACY
# ==========================================
print("\n--- 2. ACCURACY BY EVIDENCE CATEGORY ---")

cat_stats = filtered_df.groupby(['Model', 'Evidence Category'])['Is Correct'].agg(['mean', 'count']).reset_index()
cat_stats['Accuracy (%)'] = cat_stats['mean'] * 100
display(cat_stats)

Available Models: ['deepseek-r1-14b', 'gemma2-27b', 'llama3.2-latest', 'llama3.3-70b', 'mistral-7b-instruct', 'phi4-14b']
👉 Analyzing 29994 rows for models: ['deepseek-r1-14b', 'gemma2-27b', 'llama3.2-latest', 'llama3.3-70b', 'mistral-7b-instruct', 'phi4-14b']

--- 2. ACCURACY BY EVIDENCE CATEGORY ---


,Model,Evidence Category,mean,count,Accuracy (%)
0,deepseek-r1-14b,Dense Retrieval (Vector),0.330345,2119,33.034450
1,deepseek-r1-14b,KG Entity Match (Name),0.298452,2714,29.845247
2,deepseek-r1-14b,KG Traversal (Rank >5 - Supporting),0.709677,62,70.967742
3,deepseek-r1-14b,KG Traversal (Top 5 - Critical),0.528846,104,52.884615
4,gemma2-27b,Dense Retrieval (Vector),0.431884,2415,43.188406
5,gemma2-27b,KG Entity Match (Name),0.333038,2261,33.303848
6,gemma2-27b,KG Traversal (Rank >5 - Supporting),0.602740,146,60.273973
7,gemma2-27b,KG Traversal (Top 5 - Critical),0.581921,177,58.192090
8,llama3.2-latest,Dense Retrieval (Vector),0.166818,2206,16.681777
9,llama3.2-latest,KG Entity Match (Name),0.138910,2678,13.890963


In [9]:

order = ["KG Traversal (Top 5 - Critical)", "KG Traversal (Rank >5 - Supporting)", "KG Entity Match (Name)", "Dense Retrieval (Vector)", "No Evidence"]

# fig_acc = px.bar(
#     cat_stats, x="Evidence Category", y="Accuracy (%)", color="Model", barmode="group",
#     text_auto='.1f', category_orders={"Evidence Category": order},
#     title="Accuracy by Dominant Evidence Source"
# )
# fig_acc.show()

# Display Table
display(cat_stats.pivot(index="Evidence Category", columns="Model", values="Accuracy (%)").reindex(order))


Model,deepseek-r1-14b,gemma2-27b,llama3.2-latest,llama3.3-70b,mistral-7b-instruct,phi4-14b
Evidence Category,,,,,,
KG Traversal (Top 5 - Critical),52.884615,58.192090,28.888889,77.205882,26.923077,43.540670
KG Traversal (Rank >5 - Supporting),70.967742,60.273973,20.000000,64.942529,12.500000,42.950820
KG Entity Match (Name),29.845247,33.303848,13.890963,50.066934,14.348949,19.837233
Dense Retrieval (Vector),33.034450,43.188406,16.681777,41.380827,15.277778,29.201430
No Evidence,NaN,NaN,NaN,NaN,NaN,50.000000


In [12]:

# ==========================================
# 6. ANALYSIS: COMPLEXITY & ROUTES
# ==========================================
print("\n--- 3. REASONING COMPLEXITY ---")

comp_counts = filtered_df.groupby(['Model', 'Complexity']).size().reset_index(name='Count')
comp_counts['Percentage'] = comp_counts.groupby('Model')['Count'].transform(lambda x: x / x.sum() * 100)

# fig_comp = px.bar(
#     comp_counts, x="Model", y="Percentage", color="Complexity", 
#     text_auto='.1f', category_orders={"Complexity": ["Abstentions", "Simple Answers", "Complex Answers"]},
#     title="Reasoning Complexity Distribution"
# )
# fig_comp.show()

print("\n--- RAW EVIDENCE DISTRIBUTION ---")
route_data = []
for _, row in filtered_df.iterrows():
    for r in row['Used Routes']:
        route_data.append({"Model": row['Model'], "Route": r})

if route_data:
    r_df = pd.DataFrame(route_data)
    # fig_r = px.histogram(r_df, x="Model", color="Route", barmode="group", title="Total Citations by Route")
    # fig_r.show()
r_df


--- 3. REASONING COMPLEXITY ---

--- RAW EVIDENCE DISTRIBUTION ---


,Model,Route
0,llama3.3-70b,dense_def_faiss
1,llama3.3-70b,dense_def_faiss
2,llama3.3-70b,dense_def_faiss
3,llama3.3-70b,dense_def_faiss
4,llama3.3-70b,dense_def_faiss
...,...,...
69490,deepseek-r1-14b,kg_semantic_name_only
69491,deepseek-r1-14b,dense_def_faiss
69492,deepseek-r1-14b,dense_def_faiss
69493,deepseek-r1-14b,dense_def_faiss


In [13]:

# ==========================================
# 7. ANALYSIS: RANKING
# ==========================================
print("\n--- 4. TOP EVIDENCES USED ---")

complex_df = filtered_df[filtered_df['Complexity'] == "Complex Answers"]

if not complex_df.empty:
    # 1. Top-5 Usage Calculation
    for model in SELECTED_MODELS:
        m_comp = complex_df[complex_df['Model'] == model]
        if len(m_comp) > 0:
            using_top5 = m_comp['Used Ranks'].apply(lambda x: any(r <= 5 for r in x)).sum()
            perc = (using_top5 / len(m_comp)) * 100
            print(f"🔵 {model}: {perc:.1f}% of complex questions used at least one Top-5 rank.")

    # 2. Histogram
    rank_data = []
    for _, row in complex_df.iterrows():
        for r in row['Used Ranks']:
            rank_data.append({"Model": row['Model'], "Rank": r})
    r_df = pd.DataFrame(rank_data)
    # fig_hist = px.histogram(r_df, x="Rank", color="Model", barmode="overlay", nbins=32, title="Histogram of Evidence Ranks (Complex Questions)")
    # fig_hist.show()
else:
    print("No complex questions found.")



--- 4. TOP EVIDENCES USED ---
🔵 deepseek-r1-14b: 75.2% of complex questions used at least one Top-5 rank.
🔵 gemma2-27b: 75.1% of complex questions used at least one Top-5 rank.
🔵 llama3.2-latest: 72.3% of complex questions used at least one Top-5 rank.
🔵 llama3.3-70b: 53.8% of complex questions used at least one Top-5 rank.
🔵 mistral-7b-instruct: 85.4% of complex questions used at least one Top-5 rank.
🔵 phi4-14b: 82.2% of complex questions used at least one Top-5 rank.


In [14]:

# ==========================================
# 8. ANALYSIS: IMPACT & CALIBRATION
# ==========================================
print("\n--- 5. IMPACT & CALIBRATION ---")

# KG Advantage Table
impact_data = []
for model in SELECTED_MODELS:
    m_df = filtered_df[filtered_df['Model'] == model]
    is_fake = "fake" in SELECTED_TASK
    analysis_df = m_df if is_fake else m_df[~m_df['Is Abstain']]
    
    kg_acc, no_kg_acc = 0.0, 0.0
    if not analysis_df.empty:
        kg_sub = analysis_df[analysis_df['Has KG Trace']]
        no_kg_sub = analysis_df[~analysis_df['Has KG Trace']]
        if not kg_sub.empty: kg_acc = kg_sub['Is Correct'].mean() * 100
        if not no_kg_sub.empty: no_kg_acc = no_kg_sub['Is Correct'].mean() * 100
        
    impact_data.append({
        "Model": model,
        "KG Acc": kg_acc,
        "No-KG Acc": no_kg_acc,
        "KG Advantage": kg_acc - no_kg_acc
    })

display(pd.DataFrame(impact_data))



--- 5. IMPACT & CALIBRATION ---


,Model,KG Acc,No-KG Acc,KG Advantage
0,deepseek-r1-14b,70.212766,43.603812,26.608954
1,gemma2-27b,61.414791,39.911111,21.503680
2,llama3.2-latest,35.632184,28.211971,7.420213
3,llama3.3-70b,68.530021,45.956783,22.573237
4,mistral-7b-instruct,39.215686,27.178503,12.037183
5,phi4-14b,53.237410,41.275660,11.961750


In [15]:
display(analysis_df)

,Task,Model,ID,Is Correct,Is Abstain,Has KG Trace,Avg Evidence Score,Max Used Score,Score Profile,Evidence Category,Complexity,Used Ranks,Used Routes,Used Scores
38269,reasoning_fct,phi4-14b,bc8659f4-3062-4f57-9e24-e32ad92a8d4e,False,False,False,0.552561,0.663979,"[0.7771770358085632, 0.7742263078689575, 0.761...",Dense Retrieval (Vector),Complex Answers,"[13, 16, 6, 19, 21]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.5677088499069214, 0.5346575379371643, 0.663..."
38271,reasoning_fct,phi4-14b,0ac6c5c7-9826-441a-81d5-68478e6299bb,True,False,False,0.565675,0.626474,"[0.7773351669311523, 0.6264742612838745, 0.577...",KG Entity Match (Name),Complex Answers,"[4, 3, 11, 2]","[dense_def_faiss, dense_def_faiss, kg_semantic...","[0.577613115310669, 0.577613115310669, 0.48099..."
38272,reasoning_fct,phi4-14b,a6758ac9-7608-4866-bb00-c5e7b19917d5,False,False,False,0.498110,0.502303,"[0.7463321089744568, 0.6520719528198242, 0.641...",Dense Retrieval (Vector),Complex Answers,"[13, 10]","[dense_def_faiss, dense_def_faiss]","[0.4939168095588684, 0.5023033022880554]"
38274,reasoning_fct,phi4-14b,140d832a-b8ae-4791-aada-6fd62f313adb,True,False,False,0.710187,0.898582,"[0.898581862449646, 0.7858374714851379, 0.7431...",KG Entity Match (Name),Complex Answers,"[1, 9, 3, 2, 4]","[kg_semantic_name_only, dense_def_faiss, dense...","[0.898581862449646, 0.5291615128517151, 0.7431..."
38277,reasoning_fct,phi4-14b,9e5ccc5e-7716-4f99-98bd-20931299b126,False,False,False,0.625259,0.650924,"[0.7722147107124329, 0.695594072341919, 0.6707...",Dense Retrieval (Vector),Complex Answers,"[8, 5, 10, 6]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.6125761270523071, 0.6509236097335815, 0.592..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43259,reasoning_fct,phi4-14b,c83ffd59-8f3e-4d58-bae7-30ecb4c294bf,True,False,False,0.613276,0.695410,"[0.7654669880867004, 0.7222087383270264, 0.706...",Dense Retrieval (Vector),Complex Answers,"[9, 26, 12, 6, 10, 11, 4]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.6095668077468872, 0.5135133266448975, 0.605..."
43262,reasoning_fct,phi4-14b,a47d9615-bb5d-45f0-b43c-bb895dc1bd79,False,False,False,0.803504,0.803504,"[0.8035041093826294, 0.6887997984886169, 0.682...",KG Entity Match (Name),Simple Answers,[1],[kg_semantic_name_only],[0.8035041093826294]
43265,reasoning_fct,phi4-14b,127904a0-c61f-47ac-b66d-71a084e7701e,True,False,True,0.558589,0.912014,"[0.9120144844055176, 0.661913275718689, 0.5599...",KG Traversal (Rank >5 - Supporting),Complex Answers,"[1, 9, 16, 3, 19, 11, 2, 4]","[kg_semantic_name_only, dense_def_faiss, dense...","[0.9120144844055176, 0.4758324921131134, 0.436..."
43266,reasoning_fct,phi4-14b,871a0587-b48f-40a4-a4d6-116c80d420d8,False,False,False,0.653534,0.677131,"[0.790020227432251, 0.6771307587623596, 0.6732...",Dense Retrieval (Vector),Complex Answers,"[5, 9, 6, 3, 2, 4]","[dense_def_faiss, dense_def_faiss, dense_def_f...","[0.6576657295227051, 0.6021537780761719, 0.643..."


In [32]:
def is_high_confidence(row):
    scores = row['Used Scores']
    if not scores: return False
    # Count how many cited evidences have score > 0.80
    high_scores = sum(1 for s in scores if s > 0.80)
    # Check if that count is >= 60% of total citations
    return (high_scores / len(scores)) ==1

In [ ]:
scores = analysis_df['Used Scores']
high_scores = sum(1 for s in scores if s > 0.80)
high_scores

In [25]:
scored = analysis_df[analysis_df['Max Used Score'] > 0]
scored[['Is Correct','Max Used Score', 'Score Profile', 'Used Scores']]

,Is Correct,Max Used Score,Score Profile,Used Scores
38269,False,0.663979,"[0.7771770358085632, 0.7742263078689575, 0.761...","[0.5677088499069214, 0.5346575379371643, 0.663..."
38271,True,0.626474,"[0.7773351669311523, 0.6264742612838745, 0.577...","[0.577613115310669, 0.577613115310669, 0.48099..."
38272,False,0.502303,"[0.7463321089744568, 0.6520719528198242, 0.641...","[0.4939168095588684, 0.5023033022880554]"
38274,True,0.898582,"[0.898581862449646, 0.7858374714851379, 0.7431...","[0.898581862449646, 0.5291615128517151, 0.7431..."
38277,False,0.650924,"[0.7722147107124329, 0.695594072341919, 0.6707...","[0.6125761270523071, 0.6509236097335815, 0.592..."
...,...,...,...,...
43259,True,0.695410,"[0.7654669880867004, 0.7222087383270264, 0.706...","[0.6095668077468872, 0.5135133266448975, 0.605..."
43262,False,0.803504,"[0.8035041093826294, 0.6887997984886169, 0.682...",[0.8035041093826294]
43265,True,0.912014,"[0.9120144844055176, 0.661913275718689, 0.5599...","[0.9120144844055176, 0.4758324921131134, 0.436..."
43266,False,0.677131,"[0.790020227432251, 0.6771307587623596, 0.6732...","[0.6576657295227051, 0.6021537780761719, 0.643..."


In [ ]:

if not scored.empty:
    corr_score = scored[scored['Is Correct']]['Max Used Score'].mean()
    wrong_score = scored[~scored['Is Correct']]['Max Used Score'].mean()
    
    if np.isnan(corr_score): corr_score = 0
    if np.isnan(wrong_score): wrong_score = 0
    
    gap = corr_score - wrong_score
    
    # --- HIGH CONFIDENCE FAILURES (NEW LOGIC) ---


In [33]:


# Filter dataset for high-confidence questions
high_conf_total = scored[scored.apply(is_high_confidence, axis=1)]
high_conf_total[['Is Correct', 'Used Scores', 'Used Ranks']]


,Is Correct,Used Scores,Used Ranks
38323,True,[0.9331449270248413],[1]
38428,False,[0.8621128797531128],[1]
38432,False,[0.9173080921173096],[1]
38499,False,"[0.8125784397125244, 0.8069546222686768, 0.809...","[1, 3, 2]"
38523,False,[0.8277506828308105],[1]
38759,True,[0.8500000238418579],[1]
38943,False,[0.9136040806770325],[1]
38968,False,[0.8061304688453674],[1]
39064,True,[0.8943169116973877],[1]
39269,False,[0.8186072111129761],[1]


In [34]:

num_total_high = len(high_conf_total)
num_total_high

45

In [35]:
len(high_conf_total[high_conf_total['Is Correct']])

16

In [36]:

if num_total_high > 0:
    num_fails = len(high_conf_total[~high_conf_total['Is Correct']])
    fail_rate = (num_fails / num_total_high) * 100
print(fail_rate)

64.44444444444444


In [ ]:

# Calibration Chart
print("\n--- CALIBRATION CHART ---")
calib_data = []
bins = [0.0, 0.6, 0.7, 0.8, 0.9, 1.0]
labels = ['< 0.60', '0.60 - 0.69', '0.70 - 0.79', '0.80 - 0.89', '> 0.90']

for model in SELECTED_MODELS:
    m_df = filtered_df[filtered_df['Model'] == model].copy()
    if "fake" not in SELECTED_TASK: m_df = m_df[~m_df['Is Abstain']]
    
    m_df['Score Bin'] = pd.cut(m_df['Max Used Score'], bins=bins, labels=labels)
    bin_stats = m_df.groupby('Score Bin', observed=False).agg(
        Accuracy=('Is Correct', 'mean'), 
        Count=('Is Correct', 'count')
    ).reset_index()
    
    bin_stats['Accuracy'] = bin_stats['Accuracy'] * 100
    bin_stats['Model'] = model
    calib_data.append(bin_stats)

if calib_data:
    calib_df = pd.concat(calib_data)
    fig_calib = go.Figure()
    
    for model in SELECTED_MODELS:
        model_data = calib_df[calib_df['Model'] == model]
        fig_calib.add_trace(go.Bar(
            x=model_data['Score Bin'], y=model_data['Count'], 
            name=f"{model} (Count)", opacity=0.3, yaxis='y2'
        ))
        
    for model in SELECTED_MODELS:
        model_data = calib_df[calib_df['Model'] == model]
        fig_calib.add_trace(go.Scatter(
            x=model_data['Score Bin'], y=model_data['Accuracy'], 
            name=f"{model} (Accuracy %)", mode='lines+markers', line=dict(width=3)
        ))
        
    fig_calib.update_layout(
        title="Calibration: Accuracy vs. Evidence Score",
        xaxis_title="Max Evidence Score Bracket",
        yaxis=dict(title=dict(text="Accuracy (%)", font=dict(color="#1f77b4")), range=[0, 105], tickfont=dict(color="#1f77b4")),
        yaxis2=dict(title=dict(text="Number of Questions", font=dict(color="gray")), overlaying='y', side='right', showgrid=False, tickfont=dict(color="gray")),
        legend=dict(x=1.1, y=1, xanchor="left", yanchor="top", bgcolor="rgba(255,255,255,0.5)"),
        margin=dict(r=150),
        hovermode="x unified"
    )
    fig_calib.show()